In [1]:
import pandas as pd

In [2]:
pd.set_option("display.max_rows", 1000000)

# Customer

In [3]:
customer = pd.read_csv("../data/olist_customers_dataset.csv")

In [4]:
customer.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
# Check Null
customer.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [6]:
customer.count()

customer_id                 99441
customer_unique_id          99441
customer_zip_code_prefix    99441
customer_city               99441
customer_state              99441
dtype: int64

In [7]:
# Check duplicate
customer.drop_duplicates().count()

customer_id                 99441
customer_unique_id          99441
customer_zip_code_prefix    99441
customer_city               99441
customer_state              99441
dtype: int64

In [8]:
# Loại bỏ khoảng trắng, đổi thành chữ hoa đầu dòng cột customer_city
customer["customer_city"] = customer["customer_city"].str.strip().str.title()

In [9]:
# Thêm số 0 vào đầu cho các code có độ dài < 5
customer["customer_zip_code_prefix"] = customer["customer_zip_code_prefix"].astype(str) \
                                                                           .str.zfill(5) 
                                                                         

In [10]:
customer.duplicated().sum()

np.int64(0)

In [11]:
customer.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi Das Cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP


In [12]:
customer.to_csv("../clean/customers.csv", index = False, encoding = "utf-8")

# Geolocation

In [13]:
geolocation = pd.read_csv("../data/olist_geolocation_dataset.csv")

In [14]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [15]:
# Check Null
geolocation.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [16]:
geolocation.count()

geolocation_zip_code_prefix    1000163
geolocation_lat                1000163
geolocation_lng                1000163
geolocation_city               1000163
geolocation_state              1000163
dtype: int64

In [17]:
# Check Duplicate
geolocation.drop_duplicates().count()

geolocation_zip_code_prefix    738332
geolocation_lat                738332
geolocation_lng                738332
geolocation_city               738332
geolocation_state              738332
dtype: int64

In [18]:
geo_dup = geolocation.drop_duplicates()

In [19]:
import unidecode

In [20]:
# Loại bỏ khoảng trắng, đổi thành chữ hoa đầu dòng cột geolocation_city
geo_dup["geolocation_city"] = geo_dup["geolocation_city"].str.strip().str.title()
geo_dup["geolocation_city"] = geo_dup["geolocation_city"].apply(unidecode.unidecode)

In [21]:
geo_dup["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(719317)

In [22]:
geo_dup.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,Sao Paulo,SP
1,1046,-23.546081,-46.644820,Sao Paulo,SP
2,1046,-23.546129,-46.642951,Sao Paulo,SP
3,1041,-23.544392,-46.639499,Sao Paulo,SP
4,1035,-23.541578,-46.641607,Sao Paulo,SP


In [23]:
# Nhóm các bản ghi code bị dup
geo_clean = geo_dup \
                .groupby("geolocation_zip_code_prefix") \
                .agg({
                    "geolocation_lat" : "mean",
                    "geolocation_lng" : "mean", 
                    "geolocation_city" : "first",
                    "geolocation_state" : "first" 
                }) \
                .reset_index()

In [24]:
geo_clean.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550227,-46.634039,Sao Paulo,SP
1,1002,-23.547657,-46.634991,Sao Paulo,SP
2,1003,-23.549000,-46.635582,Sao Paulo,SP
3,1004,-23.549829,-46.634792,Sao Paulo,SP
4,1005,-23.549547,-46.636406,Sao Paulo,SP


In [25]:
# Thêm số 0 vào đầu cho các code có độ dài < 5
geo_clean["geolocation_zip_code_prefix"] = geo_clean["geolocation_zip_code_prefix"].astype(str) \
                                                                           .str.zfill(5) 
                                                                         

In [26]:
geo_clean.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,Sao Paulo,SP
1,01002,-23.547657,-46.634991,Sao Paulo,SP
2,01003,-23.549000,-46.635582,Sao Paulo,SP
3,01004,-23.549829,-46.634792,Sao Paulo,SP
4,01005,-23.549547,-46.636406,Sao Paulo,SP


In [27]:
geo_clean["geolocation_state"].unique()

<StringArray>
['SP', 'RJ', 'ES', 'MG', 'BA', 'SE', 'PE', 'RN', 'AL', 'PB', 'CE', 'PI', 'MA',
 'PA', 'AP', 'AM', 'RR', 'AC', 'DF', 'GO', 'RO', 'TO', 'MT', 'MS', 'PR', 'SC',
 'RS']
Length: 27, dtype: str

In [28]:
geo_clean["geolocation_city"].unique()

<StringArray>
[             'Sao Paulo',                 'Osasco',            'Carapicuiba',
                'Barueri',    'Santana De Parnaiba',  'Pirapora Do Bom Jesus',
                'Jandira',                'Itapevi',                  'Cotia',
 'Vargem Grande Paulista',
 ...
        'Ipiranga Do Sul',                'Estacao',                 'Ibiaca',
   'Santa Cecilia Do Sul',           'Vila Langaro',                'Charrua',
             'Agua Santa',                'Ciriaco',        'David Canabarro',
              'Muliterno']
Length: 5771, dtype: str

In [29]:
geo_clean = geo_clean[(geo_clean["geolocation_lat"].between(-35, 5)) &
                      (geo_clean["geolocation_lng"].between(-75, 30)) ]

In [30]:
geo_clean.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,Sao Paulo,SP
1,01002,-23.547657,-46.634991,Sao Paulo,SP
2,01003,-23.549000,-46.635582,Sao Paulo,SP
3,01004,-23.549829,-46.634792,Sao Paulo,SP
4,01005,-23.549547,-46.636406,Sao Paulo,SP


In [31]:
geo_clean.count()

geolocation_zip_code_prefix    19005
geolocation_lat                19005
geolocation_lng                19005
geolocation_city               19005
geolocation_state              19005
dtype: int64

In [52]:
geo_clean.to_csv("../clean/geolocation.csv", index = False, encoding = "utf-8")

# Order items

In [32]:
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")

In [33]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [34]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [35]:
# Check null
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [36]:
order_items.count()

order_id               112650
order_item_id          112650
product_id             112650
seller_id              112650
shipping_limit_date    112650
price                  112650
freight_value          112650
dtype: int64

In [37]:
# Check duplicate
order_items.duplicated().sum()

np.int64(0)

In [38]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [39]:
#  Chuyển đổi kiểu dữ liệu
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

In [40]:
# Lọc ra data chuẩn
order_items = order_items[(order_items["price"] > 0) & (order_items["freight_value"] > 0)]

In [41]:
order_items.count()

order_id               112267
order_item_id          112267
product_id             112267
seller_id              112267
shipping_limit_date    112267
price                  112267
freight_value          112267
dtype: int64

In [42]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [53]:
order_items.to_csv("../clean/order_items.csv", index = False, encoding = "utf-8")

# Payments

In [43]:
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")

In [44]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [45]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [46]:
# Check null
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [47]:
# Check duplicate
payments.duplicated().sum()

np.int64(0)

In [48]:
payments.count()

order_id                103886
payment_sequential      103886
payment_type            103886
payment_installments    103886
payment_value           103886
dtype: int64

In [49]:
payments["payment_type"].unique()

<StringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str

In [50]:
# Xóa khoảng trắng
payments["payment_type"] = payments["payment_type"].str.strip()

In [54]:
payments = payments[payments["payment_value"] >= 0]

In [55]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [56]:
payments.to_csv("../clean/payments.csv", index = False, encoding = "utf-8")

# Reviews

In [57]:
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

In [58]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [59]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [ ]:
# Check Null
reviews.count()

review_id                  99224
order_id                   99224
review_score               99224
review_comment_title       11568
review_comment_message     40977
review_creation_date       99224
review_answer_timestamp    99224
dtype: int64

In [ ]:
# Check Duplicate
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64